# 花书 · 第三章：概率与信息论

> 你打开的是 demo / work 副本。**学习时用 work 副本**——
> 教材副本只读（rebuild 时会被覆盖），做题请运行根目录的 `./start ch03`。

## 配套资料

- 📖 花书 PDF（vault）：`30 The Colonnade/36 Library/花书.pdf`
- 📝 朱明超精读版：`花书拆解/重要章节/3 概率与信息论.pdf`
- 🔧 Jupyter / NumPy 速查：忘了的话 [回 ch02 Sec 0](../ch02-linear-algebra/ch02.ipynb) 查
- 🎯 用途：本章是 AI-303 Workshop **Day 1 M1（信息论 + KL）+ Day 1 M2（概率基础）+ Day 2 M4（概率深入）** 的直接弹药

## 本章导航（按朱明超原文小节走，不再用 Sec 模板）

| 节 | 主题 |
|---|------|
| §1.1 | 概率与随机变量（频率派 vs 贝叶斯派） |
| §1.2 | 概率分布（PMF / PDF / CDF） |
| §1.3 | 条件概率 + 条件独立 |
| §1.4 | 期望 / 方差 / 协方差 |
| §1.5 | 七大常用分布（Bernoulli / Multinoulli / Gaussian / 多元 Gaussian / Exponential / Laplace / Dirac） |
| §1.6 | sigmoid + softplus |
| **§2** | **信息论（自信息 / 熵 / 互信息 / KL / 交叉熵）— 本章重点** |
| §3 | 图模型（贝叶斯网 + 马尔可夫网） |
| CHECKPOINT | 章末自检清单 |

## 学法

1. 每节先看 📖 **花书原文**（朱明超精读版）的定义
2. 跟着 blossom 的直觉解释 + 例题动手填 `___`
3. 跑 cell 看 `checks.assert_*` 输出 ✅ / ❌
4. **Workshop 钩子**：遇到「📍 Day X / 后续 Ch X」标记的句子，记一下名字就好——具体推导留到对应章节


In [ ]:
# 本章用到的所有库一次导入
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import scipy.stats
from utils import checks, viz

np.random.seed(0)  # 保证本章随机结果可复现

---

## §1.1 概率与随机变量


#### 📖 花书原文 — §1.1 概率与随机变量

**频率学派概率 (Frequentist Probability)**：认为概率和事件发生的频率相关。

**贝叶斯学派概率 (Bayesian Probability)**：认为概率是对某件事发生的确定程度，可以理解成是确信的程度。

**随机变量 (Random Variable)**：一个可能随机取不同值的变量。例如：抛掷一枚硬币，出现正面或者反面的结果。

### 直觉：两个学派看同一件事

**问题：抛一枚硬币正面朝上的概率是多少？**

- **频率派**：扔 10000 次硬币，看正面出现的比例——如果 ≈ 0.5，那 $P(\text{正}) = 0.5$。**概率是数出来的**。
- **贝叶斯派**：在没扔之前，「正面朝上」这件事我多确定？综合「硬币看上去均匀」「制造工艺标准」等先验，我可以说 $P(\text{正}) = 0.5$，这是我**对这件事的信念强度**。

**两派的关键区别**：频率派认为「这枚特定硬币的下一次结果」**没有概率可言**（要么正要么反，谈不上 0.5）；而贝叶斯派可以谈「这一次的概率」——因为概率是我对世界的信念。

**为什么这在深度学习里重要？**

- 训练神经网络时，我们说「这张图片是猫的概率 = 0.87」——这是**贝叶斯派**的说法
  （单张图片不能多次「重复」，但模型可以给出对它的信念强度）
- 而我们衡量模型分类准确率「在 1000 张测试图上预测对了 86%」——这是**频率派**的视角

📍 **Workshop 钩子**：贝叶斯思想在 Day 2 M6（MAP 估计）+ Day 3 M19（RLHF 的人类偏好）会反复出现——「奖励模型」可以看作对人类偏好的贝叶斯估计；KL 正则项的角色和贝叶斯先验等价。


### ✏️ 例题 1.1.E1：分辨学派

下面 4 个陈述，分别属于频率派还是贝叶斯派？先在纸上写答案再展开下面 cell。

| # | 陈述 |
|---|------|
| 1 | 这家医院过去十年新生儿男女比是 0.51 / 0.49 |
| 2 | 综合家族病史 + 基因检测，张三未来 10 年得糖尿病的概率是 0.3 |
| 3 | 这枚骰子摇 60 次，每个面大约出现 10 次，所以每面概率 1/6 |
| 4 | 训练好的分类模型给一张测试图打分 0.92，所以它「应该」是猫 |


In [ ]:
# 参考答案（判断标准：用「重复观察的比例」=频率派 / 用「主观信念 / 先验 / 单次事件」=贝叶斯派）
answers = {
    1: ('频率派', '数十年新生儿比例 = 重复观察出来的频率'),
    2: ('贝叶斯派', '单个个体的未来事件 + 综合先验信息'),
    3: ('频率派', '60 次试验的比例'),
    4: ('贝叶斯派', '单张图片不能多次重复，模型输出 = 信念强度'),
}
for i, (school, why) in answers.items():
    print(f'#{i}: {school}  —— {why}')

---

## §1.2 概率分布（PMF / PDF / CDF）


#### 📖 花书原文 — §1.2.1 概率质量函数 (PMF)

**概率质量函数 (Probability Mass Function)**：对于离散型变量，我们先定义一个随机变量，然后用 $\sim$ 符号来说明它遵循的分布：$\mathrm{x} \sim P(\mathrm{x})$，函数 $P$ 是随机变量 $\mathrm{x}$ 的 PMF。

例如，考虑一个离散型 $\mathrm{x}$ 有 $k$ 个不同的值，我们可以假设 $\mathrm{x}$ 是均匀分布的（也就是将它的每个值视为等可能的），通过将它的 PMF 设为：

$$P(\mathrm{x} = x_i) = \frac{1}{k}$$

对于所有的 $i$ 都成立。

#### 📖 花书原文 — §1.2.2 概率密度函数 (PDF)

当研究的对象是连续型时，我们可以引入同样的概念。如果一个函数 $p$ 是**概率密度函数 (Probability Density Function)**：

- 分布满足非负性条件：$\forall x \in \mathrm{x},\ p(x) \geq 0$
- 分布满足归一化条件：$\int_{-\infty}^{\infty} p(x)\,dx = 1$

例如在 $(a, b)$ 上的均匀分布：

$$U(x; a, b) = \frac{\mathbf{1}_{ab}(x)}{b - a}$$

这里 $\mathbf{1}_{ab}(x)$ 表示在 $(a, b)$ 内为 $1$，否则为 $0$。

#### 📖 花书原文 — §1.2.3 累积分布函数 (CDF)

**累积分布函数 (Cummulative Distribution Function)** 表示对小于 $x$ 的概率的积分：

$$\mathrm{CDF}(x) = \int_{-\infty}^{x} p(t)\,dt$$

In [ ]:
# 花书原文配套代码：均匀分布的 PDF + 1000 次采样直方图
from scipy.stats import uniform
fig, ax = plt.subplots(1, 1, figsize=(6, 3))
r = uniform.rvs(loc=0, scale=1, size=1000)
ax.hist(r, density=True, histtype='stepfilled', alpha=0.5, label='1000 次采样直方图')
x = np.linspace(uniform.ppf(0.01), uniform.ppf(0.99), 100)
ax.plot(x, uniform.pdf(x), 'r-', lw=3, alpha=0.8, label='uniform PDF (真值)')
ax.legend(); ax.set_title('Uniform(0, 1) — 采样直方图 vs 解析 PDF')
plt.show()

### 直觉：PMF vs PDF

**关键区别**（很多人入门时被这一点卡住）：

- **PMF 的输出就是概率**，所以 $P(\mathrm{x}=x_i) \in [0, 1]$
- **PDF 的输出不是概率**——它是「单位长度的概率密度」，**可以 $> 1$**！
  例如均匀分布 $U(0, 0.5)$ 在区间内 PDF $= 1/0.5 = 2$
- **想拿到「真正的概率」必须积分**：$P(a \le \mathrm{x} \le b) = \int_a^b p(x)\,dx$

**为什么连续型变量「单点概率 = 0」？**

对连续变量，$P(\mathrm{x} = x_0) = \int_{x_0}^{x_0} p(x)\,dx = 0$——只有**区间**才有非零概率。记住：**PDF 像质量密度（kg/m³），不是质量本身**，要乘以体积（区间长度）才得到质量（概率）。


### 可视化：把离散 vs 连续的 PMF/PDF/CDF 摆在一起


In [ ]:
from scipy.stats import bernoulli, norm
fig, axes = plt.subplots(2, 2, figsize=(10, 6))

# 离散：Bernoulli(0.3) 的 PMF + CDF
p = 0.3
axes[0, 0].bar([0, 1], bernoulli.pmf([0, 1], p), color='C0', width=0.4)
axes[0, 0].set_title('PMF: Bernoulli(0.3)')
axes[0, 0].set_xticks([0, 1]); axes[0, 0].set_ylim(0, 1)
x_disc = np.linspace(-0.5, 1.5, 100)
axes[0, 1].step(x_disc, bernoulli.cdf(x_disc, p), color='C0', where='post')
axes[0, 1].set_title('CDF: Bernoulli(0.3) — 阶梯函数')

# 连续：Normal(0, 1) 的 PDF + CDF
x_cont = np.linspace(-3, 3, 200)
axes[1, 0].plot(x_cont, norm.pdf(x_cont), color='C1', lw=2)
axes[1, 0].fill_between(x_cont, norm.pdf(x_cont), alpha=0.3, color='C1')
axes[1, 0].set_title('PDF: N(0, 1) — 光滑曲线')
axes[1, 1].plot(x_cont, norm.cdf(x_cont), color='C1', lw=2)
axes[1, 1].set_title('CDF: N(0, 1) — 单调递增 0→1')

plt.tight_layout()
plt.show()

### ✏️ 例题 1.2.E1：PMF 归一化

给定离散随机变量 $\mathrm{x}$ 取值 $\{1, 2, 3, 4\}$，PMF 形式 $P(\mathrm{x}=i) = c \cdot i$。

**任务**：找到归一化常数 $c$，让 $\sum_i P(\mathrm{x}=i) = 1$。

**提示**：所有 PMF 值加起来必须 = 1。$c \cdot (1+2+3+4) = 1$ 求 $c$。


In [ ]:
values = np.array([1, 2, 3, 4])
c = ___                             # 用 1 / 总和 算
pmf = c * values
print('c =', c)
print('PMF =', pmf)
print('sum =', pmf.sum())

checks.assert_close('1.2.E1 归一化常数 c', c, 0.1)
checks.assert_close('1.2.E1 PMF 总和=1', pmf.sum(), 1.0)

### ✏️ 例题 1.2.E2：用 CDF 算区间概率

标准正态分布 $\mathcal{N}(0, 1)$。**任务**：求 $P(-1 \le \mathrm{x} \le 1)$（落在均值 $\pm 1$ 个标准差以内的概率，应得到著名的「68%」）。

**提示**：$P(a \le \mathrm{x} \le b) = \mathrm{CDF}(b) - \mathrm{CDF}(a)$。`scipy.stats.norm.cdf(x)` 直接给标准正态的 CDF 值。


In [ ]:
from scipy.stats import norm
# 用 CDF 算区间概率：P(a ≤ x ≤ b) = CDF(b) - CDF(a)
prob = ___
print(f'P(-1 ≤ x ≤ 1) = {prob:.4f}')
checks.assert_close('1.2.E2 正态 ±1σ', prob, 0.6827, tol=1e-3)

### ✏️ 例题 1.2.E3：PDF 可以 > 1 的反直觉

构造 $U(0, 0.2)$ 的 PDF，**先猜一猜**：PDF 输出会不会 $> 1$？

**直觉**：PDF 是「密度」不是「概率」。区间 $[0, 0.2]$ 长度 0.2，总概率必须 = 1，所以 PDF $= 1 / 0.2 = 5$——大于 1 完全合法。


In [ ]:
from scipy.stats import uniform
X = uniform(loc=0, scale=0.2)        # U(0, 0.2)
print('PDF 在 x=0.1 处:', X.pdf(0.1))                  # 5.0
print('整个区间积分（区间概率）:', X.cdf(0.2) - X.cdf(0))  # 1.0

checks.assert_close('1.2.E3 PDF 值=5', X.pdf(0.1), 5.0)
checks.assert_close('1.2.E3 区间总概率=1', X.cdf(0.2) - X.cdf(0), 1.0)

---

## §1.3 条件概率与条件独立


#### 📖 花书原文 — §1.3 边缘 / 条件 / 链式法则 / 独立 / 条件独立

**边缘概率 (Marginal Probability)**：如果我们知道了一组变量的联合概率分布，但想要了解其中一个子集的概率分布。这种定义在子集上的概率分布被称为边缘概率分布：

$$\forall x \in \mathrm{x},\ P(\mathrm{x} = x) = \sum_y P(\mathrm{x} = x, \mathrm{y} = y)$$

**条件概率 (Conditional Probability)**：在很多情况下，我们感兴趣的是某个事件，在给定其他事件发生时出现的概率。这种概率叫做条件概率。我们将给定 $\mathrm{x} = x$，$\mathrm{y} = y$ 发生的条件概率记为 $P(\mathrm{y}=y \mid \mathrm{x}=x)$，可以通过下面的公式计算：

$$P(\mathrm{y} = y \mid \mathrm{x} = x) = \frac{P(\mathrm{y} = y,\, \mathrm{x} = x)}{P(\mathrm{x} = x)}$$

**条件概率的链式法则 (Chain Rule of Conditional Probability)**：任何多维随机变量的联合概率分布，都可以分解成只有一个变量的条件概率相乘的形式：

$$P(x_1, \ldots, x_n) = P(x_1) \prod_{i=2}^n P(x_i \mid x_1, \ldots, x_{i-1})$$

**独立性 (Independence)**：两个随机变量 $\mathrm{x}$ 和 $\mathrm{y}$，如果它们的概率分布可以表示成两个因子的乘积形式，并且一个因子只包含 $\mathrm{x}$ 另一个因子只包含 $\mathrm{y}$，我们就称这两个随机变量是相互独立的：

$$\forall x \in \mathrm{x},\ y \in \mathrm{y},\ p(\mathrm{x}=x, \mathrm{y}=y) = p(\mathrm{x}=x) p(\mathrm{y}=y)$$

**条件独立性 (Conditional Independence)**：如果关于 $\mathrm{x}$ 和 $\mathrm{y}$ 的条件概率分布对于 $\mathrm{z}$ 的每一个值都可以写成乘积的形式，那么这两个随机变量 $\mathrm{x}$ 和 $\mathrm{y}$ 在给定随机变量 $\mathrm{z}$ 时是条件独立的：

$$p(\mathrm{x}=x, \mathrm{y}=y \mid \mathrm{z}=z) = p(\mathrm{x}=x \mid \mathrm{z}=z) p(\mathrm{y}=y \mid \mathrm{z}=z)$$

### 直觉：从联合表格出发

下面这张表是 1000 个学生的「主修学科 (x) × 是否选修 AI 课 (y)」联合分布表（人数）：

|       | y=选 AI | y=不选 | **行总** |
|-------|--------:|------:|-------:|
| x=CS  |     320 |    80 |  **400** |
| x=Math|     150 |   150 |  **300** |
| x=Bio |      30 |   270 |  **300** |
| **列总** | **500** | **500** | **1000** |

**从这张表可以读出**：
- **联合**：$P(\mathrm{x}=\text{CS}, \mathrm{y}=\text{选}) = 320/1000 = 0.32$
- **边缘**（行/列加和）：$P(\mathrm{x}=\text{CS}) = 400/1000 = 0.4$
- **条件**：$P(\mathrm{y}=\text{选} \mid \mathrm{x}=\text{CS}) = 320/400 = 0.8$
  「在 CS 学生里，80% 选了 AI」——这就是条件概率的直觉

📍 **Workshop 钩子**：链式法则在 Day 4 写 nanoGPT 时直接用——
  语言模型把整句话的概率分解成 $P(w_1, w_2, \ldots, w_n) = \prod_i P(w_i \mid w_1, \ldots, w_{i-1})$，  这就是「自回归生成」的数学根基。


### ✏️ 例题 1.3.E1：从联合表算边缘 + 条件

用上面学科 × AI 课的表（联合 PMF 矩阵 `joint[i, j]` 表示行 i 列 j 的概率）。

**任务**：
1. 算 `P_x` ——边缘分布 $P(\mathrm{x})$（按行求和）
2. 算 `P_y_given_CS` —— 给定 $\mathrm{x}=\text{CS}$ 时 $\mathrm{y}$ 的条件分布（应 = `[0.8, 0.2]`）

**提示**：边缘 = `joint.sum(axis=1)`；条件 = 联合的某行 / 该行总和。


In [ ]:
joint = np.array([[0.32, 0.08],
                  [0.15, 0.15],
                  [0.03, 0.27]])
assert np.isclose(joint.sum(), 1.0)

P_x = ___                                     # 边缘 P(x)：joint 按行求和
P_y_given_CS = ___                            # 条件 P(y | x=CS)：第 0 行 / P_x[0]

print('P(x):              ', P_x)
print('P(y | x=CS):       ', P_y_given_CS)

checks.assert_close('1.3.E1 边缘 P(x)',     P_x, np.array([0.4, 0.3, 0.3]))
checks.assert_close('1.3.E1 P(y|x=CS)',     P_y_given_CS, np.array([0.8, 0.2]))

### ✏️ 例题 1.3.E2：链式法则验证

对 3 个 0/1 随机变量 $x_1, x_2, x_3$，用链式法则：
$$P(x_1, x_2, x_3) = P(x_1) \cdot P(x_2 \mid x_1) \cdot P(x_3 \mid x_1, x_2)$$

**任务**：给定下面的边缘 / 条件分布，重建联合分布并验证总和 = 1。


In [ ]:
# 假设：
P_x1 = np.array([0.6, 0.4])                            # P(x1=0), P(x1=1)
# P(x2 | x1)：行 = x1，列 = x2
P_x2_given_x1 = np.array([[0.7, 0.3],
                          [0.2, 0.8]])
# P(x3 | x1, x2)：[x1][x2][x3]
P_x3_given_x1x2 = np.array([[[0.9, 0.1], [0.5, 0.5]],
                            [[0.4, 0.6], [0.1, 0.9]]])

joint3 = np.zeros((2, 2, 2))
for x1 in range(2):
    for x2 in range(2):
        for x3 in range(2):
            joint3[x1, x2, x3] = (
                P_x1[x1] * P_x2_given_x1[x1, x2] * P_x3_given_x1x2[x1, x2, x3]
            )

print('联合分布 shape:', joint3.shape)
print('总和（必须=1）:', joint3.sum())
checks.assert_close('1.3.E2 链式法则总和=1', joint3.sum(), 1.0)

### ✏️ 例题 1.3.E3：判断独立 vs 条件独立

**独立的判别公式**：$\mathrm{x} \perp \mathrm{y} \iff P(x, y) = P(x) P(y)$ 对**所有** $(x, y)$ 成立。

用上面 1.3.E1 的学科 × AI 表，判断 $\mathrm{x}$（学科）和 $\mathrm{y}$（是否选 AI）是否独立。


In [ ]:
joint = np.array([[0.32, 0.08],
                  [0.15, 0.15],
                  [0.03, 0.27]])
P_x = joint.sum(axis=1)                        # 边缘 P(x)
P_y = joint.sum(axis=0)                        # 边缘 P(y)
joint_if_indep = np.outer(P_x, P_y)            # P(x)P(y) 的乘积

diff = np.abs(joint - joint_if_indep)
print('联合分布:\n', joint)
print('独立假设下应为:\n', np.round(joint_if_indep, 3))
print('最大偏差:', diff.max())

# 偏差远大于 0 → 不独立（CS 学生明显更倾向选 AI）
is_indep = diff.max() < 0.01
checks.assert_true('1.3.E3 学科与选 AI 不独立', not is_indep,
                   hint=f'最大偏差 {diff.max():.3f} 远 > 0，说明分布有相关')

---

## §1.4 随机变量的度量（期望 / 方差 / 协方差）


#### 📖 花书原文 — §1.4 期望 / 方差 / 协方差

**期望 (Expectation)**：函数 $f$ 关于概率分布 $P(\mathrm{x})$ 或 $p(\mathrm{x})$ 的期望表示为由概率分布产生 $x$，再计算 $f$ 作用到 $x$ 上后 $f(x)$ 的平均值。对于离散型随机变量，这可以通过求和得到：

$$\mathbb{E}_{\mathrm{x} \sim P}[f(x)] = \sum_x P(x) f(x)$$

对于连续型随机变量可以通过求积分得到：

$$\mathbb{E}_{\mathrm{x} \sim p}[f(x)] = \int P(x) f(x)\, dx$$

另外，**期望是线性的**：

$$\mathbb{E}_{\mathrm{x}}[\alpha f(x) + \beta g(x)] = \alpha \mathbb{E}_{\mathrm{x}}[f(x)] + \beta \mathbb{E}_{\mathrm{x}}[g(x)]$$

**方差 (Variance)**：衡量的是当我们对 $x$ 依据它的概率分布进行采样时，随机变量 $\mathrm{x}$ 的函数值会呈现多大的差异，描述采样得到的函数值在期望上下的波动程度：

$$\mathrm{Var}(f(x)) = \mathbb{E}\!\left[(f(x) - \mathbb{E}[f(x)])^2\right]$$

将方差开平方即为**标准差 (Standard Deviation)**。

**协方差 (Covariance)**：用于衡量两组值之间的**线性相关程度**：

$$\mathrm{Cov}(f(x), g(y)) = \mathbb{E}\!\left[(f(x) - \mathbb{E}[f(x)])(g(y) - \mathbb{E}[g(y)])\right]$$

**注意，独立比零协方差要求更强，因为独立还排除了非线性的相关。**

In [ ]:
# 花书原文配套代码：期望 / 方差 / 协方差
x = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9])
y = np.array([9, 8, 7, 6, 5, 4, 3, 2, 1])
Mean = np.mean(x)
Var = np.var(x)                # 默认总体方差（ddof=0）
Var_unbias = np.var(x, ddof=1) # 样本方差（无偏估计）
Cov = np.cov(x, y)
print('Mean =', Mean)
print('Var (总体) =', Var)
print('Var (无偏) =', Var_unbias)
print('Cov =\n', Cov)

### 直觉：方差为什么有「总体」和「样本」两版？

如果你**有全部数据**（整个总体），方差 = 平均「离差平方」，分母是 $N$：

$$\mathrm{Var}_{\text{总体}} = \frac{1}{N} \sum_i (x_i - \bar{x})^2$$

但实践中我们一般**只有样本**，要估计未知的「总体方差」。统计学证明：
上式会**系统性低估**总体方差（因为 $\bar{x}$ 是用样本算的，已经「最小化」了离差平方）。
正确做法是分母用 $N-1$（**Bessel 修正**）：

$$\mathrm{Var}_{\text{样本}} = \frac{1}{N-1} \sum_i (x_i - \bar{x})^2$$

NumPy 用 `ddof` 参数控制：
- `np.var(x)` = `np.var(x, ddof=0)` —— 总体方差（默认）
- `np.var(x, ddof=1)` —— 样本方差（无偏估计，推荐）

`np.cov(x, y)` 的**默认是 ddof=1**（和 var 默认相反！）。容易踩坑。


### 直觉：协方差 ≠ 独立

- **零协方差** = 无**线性**相关
- **独立** = 无**任何**形式的相关（线性 + 非线性）

经典反例：$y = x^2$，$x$ 取 $\{-1, -0.5, 0, 0.5, 1\}$（对称分布）：
- $\mathbb{E}[x] = 0$，$\mathbb{E}[xy] = \mathbb{E}[x^3] = 0$ → 协方差 = 0
- 但 $y$ 完全由 $x$ 决定 → **不独立**

**这在深度学习里的体现**：神经网络层之间往往「协方差小但高度依赖」——因为依赖关系是高度非线性的，单看 Pearson 相关系数会漏掉。


### ✏️ 例题 1.4.E1：手写期望 vs np.mean

给定离散分布 $P(\mathrm{x}=1)=0.2, P(\mathrm{x}=2)=0.5, P(\mathrm{x}=3)=0.3$。

**任务**：用定义 $\mathbb{E}[\mathrm{x}] = \sum_x P(x) \cdot x$ 手写算期望。

**提示**：直接用向量点积 `(P * values).sum()` 或 `P @ values`。


In [ ]:
values = np.array([1, 2, 3])
P = np.array([0.2, 0.5, 0.3])
assert np.isclose(P.sum(), 1.0)

E = ___                          # 用 Σ P(x) · x 手算（提示：(P * values).sum()）
print('E[x] =', E)
checks.assert_close('1.4.E1 期望', E, 2.1)

### ✏️ 例题 1.4.E2：总体方差 vs 样本方差

给一组样本 `data = [2, 4, 4, 4, 5, 5, 7, 9]`，分别用 `ddof=0` 和 `ddof=1` 算方差，并比较它们的差。

**先猜**：`ddof=1` 算出来的应该比 `ddof=0` **大**还是**小**？（提示：分母从 N=8 变成 N-1=7）


In [ ]:
data = np.array([2, 4, 4, 4, 5, 5, 7, 9], dtype=float)
var_pop    = np.var(data, ddof=0)   # 总体方差，分母 N
var_sample = np.var(data, ddof=1)   # 样本方差，分母 N-1
print(f'总体方差 (ddof=0): {var_pop:.4f}')
print(f'样本方差 (ddof=1): {var_sample:.4f}')
print(f'比值: {var_sample / var_pop:.4f}（应 = N/(N-1) = 8/7 ≈ 1.143）')

checks.assert_close('1.4.E2 总体方差', var_pop, 4.0)
checks.assert_close('1.4.E2 样本方差', var_sample, 32/7, tol=1e-4)

### ✏️ 例题 1.4.E3：零协方差 ≠ 独立（反例）

构造 $x = [-2, -1, 0, 1, 2]$，$y = x^2$。

**先猜**：协方差 $\mathrm{Cov}(x, y)$ 等于多少？

**再算 + 理解**：$y$ 由 $x$ **完全决定**（看到 $x$ 就知道 $y$），但协方差却是 0。这就是「线性无关 ≠ 独立」的经典反例。


In [ ]:
x = np.array([-2, -1, 0, 1, 2], dtype=float)
y = x ** 2

# np.cov 返回 2×2 矩阵：[[Var(x), Cov(x,y)], [Cov(x,y), Var(y)]]
cov_mat = np.cov(x, y, ddof=0)
print('协方差矩阵:\n', cov_mat)
print(f'Cov(x, y) = {cov_mat[0, 1]:.4f}')

# 协方差≈0 但 y = x^2 完全由 x 决定 → 不独立
checks.assert_close('1.4.E3 Cov(x, x²) = 0', cov_mat[0, 1], 0.0, tol=1e-10)

---

## §1.5 常用概率分布

本节按朱明超原文走 7 种分布。**学习目标**：看到分布名能立刻回忆出(1) 形状 (PDF/PMF 长什么样)、(2) 参数含义、(3) 在深度学习里典型出现场合。

下面先定义一个**通用画图 helper** `plot_distribution(X, axes)`——复刻朱明超原文的 helper，给个 scipy.stats 分布对象就画它的 PMF/PDF + CDF。


In [ ]:
# 通用分布画图 helper（inline 在本章用）
def plot_distribution(X, axes=None, label_pdf='PDF', label_cdf='CDF'):
    '''给定 scipy.stats 分布对象 X，画 PMF/PDF 和 CDF。'''
    if axes is None:
        fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    x_min, x_max = X.interval(0.99)
    x = np.linspace(x_min, x_max, 1000)
    if hasattr(X.dist, 'pdf'):                # 连续型
        axes[0].plot(x, X.pdf(x), label=label_pdf)
        axes[0].fill_between(x, X.pdf(x), alpha=0.3)
    else:                                     # 离散型
        x_int = np.unique(x.astype(int))
        axes[0].bar(x_int, X.pmf(x_int), label='PMF')
    axes[1].plot(x, X.cdf(x), label=label_cdf)
    for ax in axes:
        ax.legend()
        ax.grid(alpha=0.3)
    return axes

#### 📖 花书原文 — §1.5.1 伯努利分布 (Bernoulli) + 二项分布

**伯努利分布 (Bernoulli Distribution)** 是**单个二值随机变量**的分布，随机变量只有两种可能。它由一个参数 $\phi \in [0, 1]$ 控制，$\phi$ 给出了随机变量等于 1 的概率：

$$P(\mathrm{x}=1) = \phi,\quad P(\mathrm{x}=0) = 1 - \phi,\quad P(\mathrm{x}=x) = \phi^x (1 - \phi)^{1-x}$$

表示一次试验的结果要么成功要么失败。

📍 **Workshop 钩子**（轻量）：
- 二分类神经网络输出经过 sigmoid 后就是一个 Bernoulli 参数 $\phi$
- Day 3 M18 InstructGPT 的 Bradley-Terry reward 模型：  $P(y_w \succ y_l) = \sigma(r_w - r_l)$——这就是一个 Bernoulli


In [ ]:
from scipy.stats import bernoulli
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
p = 0.3
X = bernoulli(p)
plot_distribution(X, axes=axes)
fig.suptitle(f'Bernoulli(p={p})')
plt.show()

**Bernoulli 重复 n 次** → 二项分布 (Binomial)，统计「n 次试验里成功了几次」：

In [ ]:
# Bernoulli 一次 vs n 次 → 二项分布
p = 0.3
fig, axes = plt.subplots(1, 2, figsize=(10, 3))

# 一次试验（Bernoulli）
n_samples = 1
samples = np.random.binomial(n_samples, p, size=10000)
axes[0].bar([0, 1], [(samples == 0).mean(), (samples == 1).mean()], label='Bernoulli')
axes[0].set_title(f'Bernoulli({p}) - 1 次试验')
axes[0].legend()

# n=20 次试验（Binomial）
n_samples = 20
samples = np.random.binomial(n_samples, p, size=10000)
x_int = np.arange(0, n_samples + 1)
axes[1].bar(x_int, [(samples == k).mean() for k in x_int], label='Binomial')
axes[1].set_title(f'Binomial(n={n_samples}, p={p}) - 20 次试验里成功 k 次的概率')
axes[1].legend()
plt.show()

### ✏️ 例题 1.5.E1：Bernoulli 期望 + 方差

对 $\mathrm{x} \sim \mathrm{Bernoulli}(\phi)$，理论上有 $\mathbb{E}[\mathrm{x}] = \phi$ 和 $\mathrm{Var}(\mathrm{x}) = \phi(1-\phi)$。

**任务**：用 10000 次采样验证这两个公式（$\phi = 0.3$）。


In [ ]:
phi = 0.3
samples = np.random.binomial(1, phi, size=10000)
mean_emp = samples.mean()
var_emp = samples.var()
print(f'采样均值 = {mean_emp:.4f}，理论 φ = {phi}')
print(f'采样方差 = {var_emp:.4f}，理论 φ(1-φ) = {phi * (1-phi):.4f}')

checks.assert_close('1.5.E1 Bernoulli 均值', mean_emp, phi, tol=0.02)
checks.assert_close('1.5.E1 Bernoulli 方差', var_emp, phi * (1 - phi), tol=0.02)

#### 📖 花书原文 — §1.5.2 范畴分布 (Multinoulli / Categorical) + 多项分布

**范畴分布 (Multinoulli Distribution)** 是指在**具有 $k$ 个不同值**的单个离散型随机变量上的分布：

$$p(\mathrm{x} = x) = \prod_i \phi_i^{x_i}$$

例如每次试验的结果就可以记为一个 $k$ 维的向量，只有此次试验的结果对应的维度记为 1，其他记为 0。

📍 **Workshop 钩子**（重点！）：
- **语言模型的输出就是 Categorical 分布**：在词表的 $|V| \approx 50000$ 个 token 上做选择
- 神经网络最后一层 softmax 之后得到的就是 Categorical 分布的 $\phi$
- Day 2 M8 + Day 3 M21 nanoGPT 训练时，每个位置的 loss = $-\log P(\text{下一 token} \mid \text{前文})$，  这就是 Categorical 分布的负对数似然


In [ ]:
# Categorical 一次 vs n 次 → 多项分布
k = 5                                             # 5 个类别（比如骰子去掉一面）
phi = np.array([0.1, 0.2, 0.3, 0.3, 0.1])         # 概率向量
assert np.isclose(phi.sum(), 1.0)

fig, axes = plt.subplots(1, 2, figsize=(10, 3))

# 一次试验：抽 1 个，每次只有 1 个 bin 为 1
n_trials = 1
sample = np.random.multinomial(n_trials, phi)
axes[0].bar(range(k), sample, label=f'Multinoulli 1次 → {sample}')
axes[0].set_title('Multinoulli 一次：只有 1 个 bin 为 1')
axes[0].legend()

# n=1000 次：成多项分布，估计真实 phi
n_trials = 1000
samples = np.random.multinomial(n_trials, phi)
axes[1].bar(range(k), samples / n_trials, label='1000 次频率')
axes[1].bar(range(k), phi, alpha=0.3, color='red', label='真值 φ')
axes[1].set_title(f'Multinomial({n_trials} 次) 频率 vs 真值')
axes[1].legend()
plt.show()

### ✏️ 例题 1.5.E2：softmax → Categorical 概率

神经网络输出一组未归一化的「logits」$z$，要变成 Categorical 分布的概率向量，用 $\mathrm{softmax}(z)_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$。

**任务**：给定 $z = [2.0, 1.0, 0.1, -1.0]$（4 类的 logits），算 softmax 后的概率。

**数值稳定 trick**：先减去 $\max(z)$ 再 exp（防止 $e^{z_i}$ 上溢）。


In [ ]:
def softmax(z):
    # 提示：(1) 减 max 数值稳定 (2) np.exp (3) 除以和
    z = z - z.max()
    e = ___
    return ___

z = np.array([2.0, 1.0, 0.1, -1.0])
p = softmax(z)
print('softmax(z) =', p)
print('总和 =', p.sum())

expected = np.array([0.6381, 0.2347, 0.0954, 0.0318])
checks.assert_close('1.5.E2 softmax 概率', p, expected, tol=1e-3)
checks.assert_close('1.5.E2 总和=1', p.sum(), 1.0)

#### 📖 花书原文 — §1.5.3 高斯分布 (Gaussian / Normal)

**高斯分布 (Gaussian Distribution)** 或正态分布 (Normal Distribution) 形式如下：

$$\mathcal{N}(x; \mu, \sigma^2) = \sqrt{\frac{1}{2\pi\sigma^2}} \exp\!\left(-\frac{1}{2\sigma^2}(x - \mu)^2\right)$$

有时也会用 $\beta = \frac{1}{\sigma^2}$ 表示分布的精度 (precision)。**中心极限定理 (Central Limit Theorem)** 认为，大量的独立随机变量的和近似于一个正态分布，因此可以认为噪声是属于正态分布的。

📍 **Workshop 钩子**：
- NN 权重初始化普遍用 Gaussian（He / Xavier 初始化都基于 Gaussian 调方差）
- Day 2 M9 Adam 优化器内部假设梯度分量近似 Gaussian
- Day 5 Mini-DPO 训练时如果加入「探索噪声」也用 Gaussian


In [ ]:
from scipy.stats import norm
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
mu, sigma = 0, 1
X = norm(mu, sigma)         # 标准正态
plot_distribution(X, axes=axes)
fig.suptitle(f'N(μ={mu}, σ={sigma})')
plt.show()

### ✏️ 例题 1.5.E3：高斯分布的 68/95/99.7 法则

对标准正态 $\mathcal{N}(0, 1)$，「$\mu \pm k\sigma$ 区间」的概率：
- $k=1$ → ~68.3%
- $k=2$ → ~95.4%
- $k=3$ → ~99.7%

**任务**：用 `norm.cdf` 验证这三个数值。


In [ ]:
from scipy.stats import norm
for k in [1, 2, 3]:
    prob = norm.cdf(k) - norm.cdf(-k)
    print(f'P(|x| ≤ {k}σ) = {prob:.4f}  →  {prob*100:.2f}%')

checks.assert_close('1.5.E3 ±1σ', norm.cdf(1) - norm.cdf(-1), 0.6827, tol=1e-3)
checks.assert_close('1.5.E3 ±2σ', norm.cdf(2) - norm.cdf(-2), 0.9545, tol=1e-3)
checks.assert_close('1.5.E3 ±3σ', norm.cdf(3) - norm.cdf(-3), 0.9973, tol=1e-3)

#### 📖 花书原文 — §1.5.4 多元高斯分布 (Multivariate Normal)

多元正态分布 (Multivariate Normal Distribution) 形式如下：

$$\mathcal{N}(x; \mu, \Sigma) = \sqrt{\frac{1}{(2\pi)^n \det(\Sigma)}} \exp\!\left(-\frac{1}{2}(x - \mu)^\top \Sigma^{-1} (x - \mu)\right)$$

其中 $\mu \in \mathbb{R}^n$ 是均值向量，$\Sigma \in \mathbb{R}^{n \times n}$ 是协方差矩阵（正定）。

📍 **Workshop 钩子**：
- VAE / 扩散模型的 latent space 假设多元 Gaussian
- 协方差矩阵 $\Sigma$ 的特征向量 = 数据的主成分（回 Ch 2 PCA 那条线）


In [ ]:
# 朱明超原文配套：2D 多元正态的等高线
from scipy.stats import multivariate_normal
x, y = np.mgrid[-1:1:.01, -1:1:.01]
pos = np.dstack((x, y))
fig = plt.figure(figsize=(5, 5))
ax = fig.add_subplot(111)
mu = [0.5, -0.2]                  # 均值
sigma = [[2.0, 0.3], [0.3, 0.5]]  # 协方差矩阵
X = multivariate_normal(mu, sigma)
cs = ax.contourf(x, y, X.pdf(pos), cmap='viridis')
ax.scatter(*mu, c='red', s=50, label='μ', edgecolors='white')
ax.set_title(f'2D Gaussian, μ={mu}, Σ={sigma}')
ax.legend()
plt.colorbar(cs)
plt.show()

#### 📖 花书原文 — §1.5.5 指数分布 (Exponential)

指数分布 (Exponential Distribution) 形式如下：

$$p(x; \lambda) = \lambda \mathbf{1}_{x \geq 0} \exp(-\lambda x)$$

是用于在 $x = 0$ 处获得最高的概率的分布，其中 $\lambda > 0$ 是分布的一个参数，常被称为率参数 (Rate Parameter)。

In [ ]:
from scipy.stats import expon
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
X = expon(scale=1)                # scale = 1/λ
plot_distribution(X, axes=axes)
fig.suptitle('Exponential(λ=1)')
plt.show()

#### 📖 花书原文 — §1.5.6 拉普拉斯分布 (Laplace)

拉普拉斯分布 (Laplace Distribution) 形式如下：

$$\mathrm{Laplace}(x; \mu, \gamma) = \frac{1}{2\gamma} \exp\!\left(-\frac{|x - \mu|}{\gamma}\right)$$

这也是可以在一个点获得比较高的概率的分布。

📍 **Workshop 钩子**：拉普拉斯分布的 NLL = $|x - \mu|/\gamma + \text{常数}$——
这就是 **L1 范数损失**的来源（相对的，Gaussian NLL = $L2$ 损失）。等到 Ch 7 讲正则化时会再相遇。


In [ ]:
from scipy.stats import laplace
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
mu, gamma = 0, 1
X = laplace(loc=mu, scale=gamma)
plot_distribution(X, axes=axes)
fig.suptitle(f'Laplace(μ={mu}, γ={gamma})')
plt.show()

### ✏️ 例题 1.5.E4：Laplace vs Gaussian 的「尾巴」

Laplace 比 Gaussian 有**更厚的尾巴**（远离均值处概率更大）。
**任务**：算「$|x| > 3$」在两种分布下的概率（都用 $\mu=0, \sigma=1$，Laplace 用 $\gamma = 1/\sqrt{2}$ 让方差相同）。


In [ ]:
from scipy.stats import norm, laplace
X_norm = norm(0, 1)
X_lap = laplace(loc=0, scale=1/np.sqrt(2))   # 方差 = 2γ² = 1

tail_norm = 2 * (1 - X_norm.cdf(3))           # P(|x| > 3) under Gaussian
tail_lap  = 2 * (1 - X_lap.cdf(3))            # under Laplace
print(f'Gaussian 尾巴概率: {tail_norm:.5f}')
print(f'Laplace  尾巴概率: {tail_lap:.5f}')
print(f'比值（Laplace 厚多少）: {tail_lap / tail_norm:.2f}x')

checks.assert_true('1.5.E4 Laplace 尾巴更厚', tail_lap > tail_norm)

#### 📖 花书原文 — §1.5.7 Dirac 分布 + 经验分布

**Dirac delta 函数** 定义为 $p(x) = \delta(x - \mu)$，这是一个泛函数。它常被用于组成**经验分布 (Empirical Distribution)**：

$$\hat{p}(x) = \frac{1}{m} \sum_{i=1}^m \delta(x - x^{(i)})$$

**直觉**：Dirac 函数把「全部概率压在一个点上」。经验分布就是把训练集每个数据点看成一个 Dirac，然后均匀分配概率——这就是「训练数据本身」的概率视角。

📍 **Workshop 钩子**（重要！）：
- Day 2 M5 推 **MLE = 最小化交叉熵**时，$\hat{p}_{\text{data}}$（真实数据分布）就用经验分布近似——一堆 Dirac 函数加起来
- $\theta_{\mathrm{MLE}} = \arg\min_\theta \mathbb{E}_{x \sim \hat{p}_{\text{data}}}\!\left[-\log p_{\text{model}}(x;\theta)\right]$ —— 对经验分布求期望 = 对训练样本求平均


In [ ]:
# 用直方图模拟「经验分布」（无法直接画 δ，但 1000 个样本的直方图就是它的可视化）
samples = np.random.normal(0, 1, 50)               # 假装真实数据
fig, ax = plt.subplots(figsize=(7, 3))
ax.scatter(samples, np.zeros_like(samples), marker='|', s=200,
           color='black', label=f'经验分布: 50 个 δ 函数')
ax.axhline(0, color='gray', alpha=0.3)
ax.set_title('经验分布 = 每个数据点一个 δ，权重 1/N')
ax.set_yticks([])
ax.legend()
plt.show()

---

## §1.6 常用函数：sigmoid + softplus


#### 📖 花书原文 — §1.6.1 logistic sigmoid 函数

$$\sigma(x) = \frac{1}{1 + \exp(-x)}$$

logistic sigmoid 函数通常用来产生伯努利分布中的参数 $\phi$，因为它的范围是 $(0, 1)$，处在 $\phi$ 的有效取值范围内。sigmoid 函数在变量取绝对值非常大的正值或负值时会出现**饱和 (Saturate)** 现象，意味着函数会变得很平，并且对输入的微小改变会变得不敏感。

#### 📖 花书原文 — §1.6.2 softplus 函数

$$\zeta(x) = \log(1 + \exp(x))$$

softplus 函数可以用来产生正态分布的 $\beta$ 和 $\sigma$ 参数，因为它的范围是 $(0, \infty)$。当处理包含 sigmoid 函数的表达式时它也经常出现。softplus 函数名来源于它是另外一个函数的平滑（或「软化」）形式，这个函数是：

$$x^+ = \max(0, x)$$

In [ ]:
# 花书原文配套代码：sigmoid + softplus 画图
x = np.linspace(-10, 10, 100)
sigmoid = 1 / (1 + np.exp(-x))
softplus = np.log(1 + np.exp(x))
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(x, sigmoid, label='sigmoid σ(x)')
axes[0].axhline(0, color='gray', alpha=0.3)
axes[0].axhline(1, color='gray', alpha=0.3, linestyle='--', label='饱和上限')
axes[0].legend(); axes[0].set_title('sigmoid: (-∞, ∞) → (0, 1)')
axes[1].plot(x, softplus, label='softplus ζ(x)')
axes[1].plot(x, np.maximum(0, x), '--', alpha=0.5, label='max(0, x) = ReLU')
axes[1].legend(); axes[1].set_title('softplus: (-∞, ∞) → (0, ∞)，是 ReLU 的「软化」')
plt.show()

### 关键性质（记一下，后面会反复用）

$$1 - \sigma(x) = \sigma(-x) \qquad \frac{d}{dx}\sigma(x) = \sigma(x)(1 - \sigma(x))$$

$$\frac{d}{dx}\zeta(x) = \sigma(x) \qquad \zeta(x) - \zeta(-x) = x$$

**直觉**：sigmoid 把整条实数轴「挤压」进 $(0, 1)$——任何 logit / score 都能拿来做概率。

📍 **Workshop 钩子**（重要！）：

**sigmoid 是 DPO 推导的起点之一**。Day 3 M18 InstructGPT 用 **Bradley-Terry 模型** 训练 reward model：

$$P(y_w \succ y_l \mid x) = \sigma(r_\phi(x, y_w) - r_\phi(x, y_l))$$

「在给定 prompt $x$ 下，输出 $y_w$ 比 $y_l$ 好」的概率 = **sigmoid 作用在两个 reward 之差上**。

Day 4 M26 的 DPO loss 就是从这条公式 + KL 约束推出来的——
**等你学到 Day 4 再回来看，这一节的 sigmoid 是核心零件**。


### ✏️ 例题 1.6.E1：验证 σ(-x) = 1 - σ(x)

**任务**：手写 sigmoid，对 100 个随机 $x$ 验证 $\sigma(-x) + \sigma(x) = 1$。


In [ ]:
def sigmoid(x):
    # 标准定义 σ(x) = 1 / (1 + exp(-x))
    return ___

np.random.seed(0)
x = np.random.randn(100) * 5
sum_pair = sigmoid(x) + sigmoid(-x)
print(f'σ(x) + σ(-x) 最大偏差离 1: {np.abs(sum_pair - 1).max():.2e}')

checks.assert_close('1.6.E1 σ(x)+σ(-x)=1', sum_pair, np.ones(100), tol=1e-10)

### ✏️ 例题 1.6.E2：sigmoid 的数值稳定问题

对很大的负 $x$（比如 $x = -1000$），$\exp(-x) = \exp(1000)$ 会**上溢**到 `inf`。

**先猜**：naive 实现 `1 / (1 + np.exp(-x))` 在 $x = -1000$ 时会给出什么？
**任务**：用 `np.where` 写一个数值稳定版本——$x \geq 0$ 用原公式，$x < 0$ 用等价的 $e^x / (1 + e^x)$。


In [ ]:
def sigmoid_stable(x):
    return np.where(
        x >= 0,
        1.0 / (1.0 + np.exp(-x)),       # 大正数情况：分母不会爆
        np.exp(x) / (1.0 + np.exp(x))   # 大负数情况：分子分母都很小
    )

import warnings
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    # naive 在极端 x 会上溢
    naive = 1.0 / (1.0 + np.exp(-np.array([-1000.0, 0.0, 1000.0])))
stable = sigmoid_stable(np.array([-1000.0, 0.0, 1000.0]))
print('naive  (x=-1000, 0, 1000):', naive)    # 可能 nan / overflow warning
print('stable (x=-1000, 0, 1000):', stable)   # 干净的 [0, 0.5, 1]

checks.assert_close('1.6.E2 stable @x=0', stable[1], 0.5)
checks.assert_close('1.6.E2 stable @x=-1000', stable[0], 0.0)
checks.assert_close('1.6.E2 stable @x=1000', stable[2], 1.0)

### ✏️ 例题 1.6.E3：softplus 是 sigmoid 的反导

性质：$\zeta'(x) = \sigma(x)$。

**任务**：用 `np.gradient` 数值求 softplus 的导数，和 sigmoid 直接计算的对比，验证它们近似相等。


In [ ]:
x = np.linspace(-5, 5, 1000)
softplus = np.log1p(np.exp(-np.abs(x))) + np.maximum(x, 0)   # 数值稳定版
dsoftplus_dx = np.gradient(softplus, x)                       # 数值梯度
sigmoid_x = 1.0 / (1.0 + np.exp(-x))

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(x, dsoftplus_dx, label="数值微分: d/dx softplus", lw=3)
ax.plot(x, sigmoid_x, '--', label='σ(x) 直接计算', lw=2)
ax.legend(); ax.set_title('验证: ζ\'(x) = σ(x)')
plt.show()

max_err = np.abs(dsoftplus_dx - sigmoid_x).max()
print(f'最大偏差: {max_err:.6f}')
checks.assert_true('1.6.E3 ζ\'(x) ≈ σ(x)', max_err < 0.01)

---

# §2 信息论 — 本章重点

这一节是 AI-303 Workshop **Day 1 M1** 直接弹药——
KL 散度 + 交叉熵是后面 RLHF/DPO 推导的两块基石。学到这里能默写公式 = 后面省 10 倍力气。


#### 📖 花书原文 — §2 信息论（自信息 / 熵 / 联合熵 / 条件熵 / 互信息）

**信息论背后的思想：一件不太可能的事件比一件比较可能的事件更有信息量。**

信息 (Information) 需要满足的三个条件：

- 比较可能发生的事件的信息量要少。
- 比较不可能发生的事件的信息量要大。
- 独立发生的事件之间的信息量应该是可以叠加的。例如，投掷的硬币两次正面朝上传递的信息量，应该是投掷一次硬币正面朝上的信息量的两倍。

**自信息 (Self-Information)**：对事件 $\mathrm{x} = x$，我们定义：

$$I(x) = -\log P(x)$$

自信息满足上面三个条件，单位是奈特 (nats)（底为 $e$）。

**香农熵 (Shannon Entropy)**：上述的自信息只包含一个事件的信息，而**对于整个概率分布 $P$**，不确定性可以这样衡量：

$$\mathbb{E}_{x \sim P}[I(x)] = -\mathbb{E}_{x \sim P}[\log P(x)]$$

也可以表示成 $H(P)$。香农熵是编码原理中最优编码长度。

**多个随机变量**：

- **联合熵 (Joint Entropy)**：表示同时考虑多个事件的条件下（即考虑联合分布概率）的熵。

$$H(X, Y) = -\sum_{x, y} P(x, y) \log(P(x, y))$$

- **条件熵 (Conditional Entropy)**：表示某件事情已经发生的情况下，另外一件事情的熵。

$$H(X \mid Y) = -\sum_y P(y) \sum_x P(x \mid y) \log(P(x \mid y))$$

- **互信息 (Mutual Information)**：表示两个事件的信息**相交**的部分。

$$I(X, Y) = H(X) + H(Y) - H(X, Y)$$

- **信息变差 (Variation of Information)**：表示两个事件的信息**不相交**的部分。

$$V(X, Y) = H(X, Y) - I(X, Y)$$

### 直觉：为什么 $-\log P$？

你想要一个「信息量」函数 $I(x)$ 满足上面三个条件。**独立事件叠加**那条最关键：$P(x, y) = P(x) P(y) \Rightarrow I(x, y) = I(x) + I(y)$。

什么函数能把「乘法」变成「加法」？$\log$。

再要求「越罕见信息量越大」（小概率 → 大信息），所以加负号 → $I(x) = -\log P(x)$。唯一选择，没有任何 ad-hoc。

**单位**：以 $e$ 为底叫 **nats**（自然单位，深度学习里用），以 2 为底叫 **bits**（通信学里用）。


### 可视化：二值分布的熵 $H(p)$ —— 朱明超原文图 7

In [ ]:
# 复刻朱明超原文图：H(p) = -p log p - (1-p) log(1-p)
p = np.linspace(1e-6, 1 - 1e-6, 100)
entropy = -p * np.log(p) - (1 - p) * np.log(1 - p)
plt.figure(figsize=(5, 4))
plt.plot(p, entropy)
plt.axvline(0.5, color='red', linestyle='--', alpha=0.5, label='p=0.5 → 最大熵 ln 2 ≈ 0.693')
plt.xlabel('p (Bernoulli 参数)')
plt.ylabel('Shannon entropy H(p) [nats]')
plt.title('二值分布的熵: 最不确定 = 50/50')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**看图直觉**：
- $p = 0$ 或 $p = 1$ → 熵 = 0（完全确定，没有信息）
- $p = 0.5$ → 熵最大 $= \ln 2 \approx 0.693$ nats（最不确定）
- **熵 = 描述这个分布所需的「最少平均比特数」**

### ✏️ 例题 2.E1：手写自信息 + 验证罕见性

**任务**：算下面 4 个事件的自信息（nats），验证「越罕见信息量越大」。

| 事件 | 概率 |
|------|------|
| 太阳明天升起 | 0.9999999 |
| 抛硬币正面 | 0.5 |
| 你今天买彩票中头奖 | 1e-7 |
| 不可能事件（占位） | 1e-12 |


In [ ]:
def self_info(p):
    # 自信息 I(x) = -log P(x)
    return ___

events = [('太阳明天升起', 0.9999999),
          ('抛硬币正面', 0.5),
          ('彩票头奖', 1e-7),
          ('几乎不可能', 1e-12)]
for name, p in events:
    print(f'{name:<15s} p={p:<12.1e}  I(x) = {self_info(p):.3f} nats')

checks.assert_true('2.E1 罕见事件 I 更大', self_info(1e-7) > self_info(0.5))

### ✏️ 例题 2.E2：手写香农熵 + scipy 对比

**任务**：手写 $H(P) = -\sum_i P_i \log P_i$，对几个分布算熵，并用 `scipy.stats.entropy` 验证。

**提示**：注意 $P_i = 0$ 时 $0 \log 0 \to 0$（约定，避免 NaN），用 `np.where` 处理。


In [ ]:
def shannon_entropy(P):
    P = np.asarray(P, dtype=float)
    # 实现 H(P) = -Σ P_i log P_i，注意 P_i=0 时贡献 = 0
    return ___

from scipy.stats import entropy
for name, P in [
    ('均匀 [0.25]*4', [0.25, 0.25, 0.25, 0.25]),
    ('偏分布 [0.7,0.2,0.1]', [0.7, 0.2, 0.1]),
    ('确定 [1,0,0]', [1.0, 0.0, 0.0]),
]:
    h_mine = shannon_entropy(P)
    h_scipy = entropy(P)
    print(f'{name:<25s} 手写={h_mine:.4f}  scipy={h_scipy:.4f}')
    checks.assert_close(f'2.E2 H({name})', h_mine, h_scipy)

#### 📖 花书原文 — §2 KL 散度 + 交叉熵

**KL 散度 (Kullback-Leibler Divergence)** 用于衡量两个分布 $P(\mathrm{x})$ 和 $Q(\mathrm{x})$ 之间的差距：

$$D_{\mathrm{KL}}(P \| Q) = \mathbb{E}_{x \sim P}\!\left[\log \frac{P(x)}{Q(x)}\right] = \mathbb{E}_{x \sim P}\!\left[\log P(x) - \log Q(x)\right]$$

注意 $D_{\mathrm{KL}}(P\|Q) \neq D_{\mathrm{KL}}(Q\|P)$，**不满足对称性**。

**交叉熵 (Cross Entropy)**：

$$H(P, Q) = H(P) + D_{\mathrm{KL}}(P \| Q) = -\mathbb{E}_{x \sim P}[\log Q(x)]$$

假设 $P$ 是真实分布，$Q$ 是模型分布，那么**最小化交叉熵 $H(P, Q)$ 可以让模型分布逼近真实分布**。

### 直觉：KL 散度的两个视角

**视角 1**（编码视角）：KL 是「用 $Q$ 的最优编码去编码来自 $P$ 的样本，比用 $P$ 自己的最优编码多用多少比特」。$Q$ 越像 $P$，多用的比特越少；$Q$ 完全等于 $P$ → KL = 0。

**视角 2**（似然比期望）：KL = $\mathbb{E}_P\!\left[\log\frac{P}{Q}\right]$ —— 在真实分布 $P$ 下，「真实概率 / 模型概率」的对数比的平均值。$Q$ 在 $P$ 高概率处给的概率越小，KL 越大。

**关键性质**：
- $D_{\mathrm{KL}}(P\|Q) \geq 0$，等号当且仅当 $P = Q$（Gibbs' inequality）
- **不对称**：$D_{\mathrm{KL}}(P\|Q) \neq D_{\mathrm{KL}}(Q\|P)$ —— 这意味着「用 $Q$ 拟合 $P$」和「用 $P$ 拟合 $Q$」**目标不同**


### 📍 Workshop 钩子（**必读**，本章最重要的两条桥接）

1. **RLHF 的 KL penalty**（Day 3 M18 InstructGPT）：
$$\max_{\pi_\theta} \mathbb{E}_{x, y \sim \pi_\theta}[r_\phi(x, y)] - \beta\, D_{\mathrm{KL}}[\pi_\theta \| \pi_{\text{ref}}]$$
「让奖励高，但别离参考策略太远」——KL 是「弹簧」，$\beta$ 是弹性系数。这条公式没 KL 散度就不存在。

2. **MLE = 最小化交叉熵**（Day 2 M5 推导核心）：
$$\theta_{\mathrm{MLE}} = \arg\min_\theta H(\hat{p}_{\text{data}}, p_{\text{model}}) = \arg\min_\theta \mathbb{E}_{x \sim \hat{p}_{\text{data}}}\!\left[-\log p_{\text{model}}(x; \theta)\right]$$
训练任何一个神经网络分类器/语言模型，loss = 交叉熵 = MLE 等价。Day 4 写 nanoGPT 训练循环，loss 函数就是它。

3. **DPO 推导依赖正向 vs 反向 KL 不对称**（Day 4 M26）：知道这个区别能让你后面读 DPO 论文更轻松。

### ✏️ 例题 2.E3：手写 KL 散度 + scipy 对比

**任务**：实现 $D_{\mathrm{KL}}(P\|Q) = \sum_i P_i \log \frac{P_i}{Q_i}$，并和 `scipy.stats.entropy(P, Q)` 对比（scipy 的 `entropy(P, Q)` 就是 $D_{\mathrm{KL}}(P\|Q)$）。

**约定**：$P_i = 0$ 时该项贡献 = 0。


In [ ]:
def kl_divergence(P, Q):
    P = np.asarray(P, dtype=float)
    Q = np.asarray(Q, dtype=float)
    # 实现 Σ P_i · log(P_i / Q_i)，P_i=0 时贡献 = 0
    return ___

from scipy.stats import entropy
P = np.array([0.1, 0.4, 0.5])
Q = np.array([0.2, 0.3, 0.5])

kl_mine = kl_divergence(P, Q)
kl_scipy = entropy(P, Q)
print(f'D_KL(P || Q) = {kl_mine:.6f}（scipy: {kl_scipy:.6f}）')
checks.assert_close('2.E3 KL 散度', kl_mine, kl_scipy)
checks.assert_close('2.E3 KL(P||P)=0', kl_divergence(P, P), 0.0)

### ✏️ 例题 2.E4：验证 $H(P, Q) = H(P) + D_{\mathrm{KL}}(P\|Q)$

**任务**：分别算交叉熵 $H(P, Q) = -\sum P_i \log Q_i$、香农熵 $H(P)$、KL $D_{\mathrm{KL}}(P\|Q)$，验证恒等式 $H(P, Q) = H(P) + D_{\mathrm{KL}}(P\|Q)$。


In [ ]:
def cross_entropy(P, Q):
    P = np.asarray(P, dtype=float)
    Q = np.asarray(Q, dtype=float)
    return -np.sum(np.where(P > 0, P * np.log(Q), 0.0))

P = np.array([0.1, 0.4, 0.5])
Q = np.array([0.2, 0.3, 0.5])

H_P = shannon_entropy(P)
H_PQ = cross_entropy(P, Q)
KL = kl_divergence(P, Q)
print(f'H(P) = {H_P:.6f}')
print(f'D_KL(P||Q) = {KL:.6f}')
print(f'H(P,Q) = {H_PQ:.6f}（应 = H(P) + KL = {H_P + KL:.6f}）')

checks.assert_close('2.E4 H(P,Q)=H(P)+KL', H_PQ, H_P + KL)

### ✏️ 例题 2.E5：KL 不对称——朱明超原文图 9 复刻（重要！）

**场景**：真实分布 $P$ 是双峰高斯混合（如 $\mathcal{N}(3, 0.5) + \mathcal{N}(6, 0.5)$）。
用**单峰**高斯 $Q$ 去拟合，分别最小化两种 KL：

- **正向 KL** $\arg\min_q D_{\mathrm{KL}}(P\|Q)$ → $Q$ 倾向**覆盖** $P$ 的所有峰（mean-seeking）
- **反向 KL** $\arg\min_q D_{\mathrm{KL}}(Q\|P)$ → $Q$ 倾向**挑一个峰**（mode-seeking）

**这就是 DPO 推导里反复出现的「KL 不对称」现象的根源**。


In [ ]:
from scipy.stats import norm, entropy
from scipy.integrate import trapezoid    # NumPy 2.0+ 移走了 np.trapz，用 scipy 更稳

# 真实双峰分布 p(x)
x = np.linspace(1, 8, 500)
p = norm.pdf(x, 3, 0.5) + norm.pdf(x, 6, 0.5)
p = p / trapezoid(p, x)             # 归一化

# 在所有 (μ, σ) 网格上找正向 / 反向 KL 最小的 q
KL_pq, KL_qp, q_list = [], [], []
for mu in np.linspace(0, 10, 50):
    for sigma in np.linspace(0.1, 5, 50):
        q = norm.pdf(x, mu, sigma)
        q = q / trapezoid(q, x)      # 归一化
        q_list.append(q)
        KL_pq.append(trapezoid(np.where(p > 0, p * np.log(p / (q + 1e-12)), 0), x))
        KL_qp.append(trapezoid(np.where(q > 0, q * np.log(q / (p + 1e-12)), 0), x))
q_pq_min = q_list[int(np.argmin(KL_pq))]
q_qp_min = q_list[int(np.argmin(KL_qp))]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(x, p, 'b', label='p(x) 真实双峰', lw=2)
axes[0].plot(x, q_pq_min, 'g--', label='q*(x)', lw=2)
axes[0].set_title('正向 KL：q* = argmin D_KL(p||q) → 覆盖型')
axes[0].legend()
axes[1].plot(x, p, 'b', label='p(x) 真实双峰', lw=2)
axes[1].plot(x, q_qp_min, 'g--', label='q*(x)', lw=2)
axes[1].set_title('反向 KL：q* = argmin D_KL(q||p) → 挑峰型')
axes[1].legend()
plt.suptitle('KL 散度的不对称性：决定了 q 的拟合策略')
plt.show()
print('→ 正向 KL 选了「胖单峰」覆盖两个真峰（mean-seeking / 0-avoiding）')
print('→ 反向 KL 选了「瘦单峰」挑一个真峰（mode-seeking / 0-forcing）')

### ✏️ 例题 2.E6：演示「最优编码 = 熵」（朱明超原文示例）

对一段 ASCII 文本，逐字符算频率，再算 $H = -\sum p_i \log_2 p_i$（**底为 2** → 单位 bits）。
这就是这段文本用最优编码每字符**至少**需要的比特数。

**任务**：复刻朱明超原文 H 函数，对随机文本验证 H ≈ log₂(可能字符数)。


In [ ]:
import math, random

def H_bits(sentence):
    entropy = 0.0
    for c in range(256):
        Px = sentence.count(chr(c)) / len(sentence)
        if Px > 0:
            entropy += -Px * math.log(Px, 2)   # 底 2 → bits
    return entropy

# 用 64 个不同字符随机生成长文本——理论熵应 ≈ log2(64) = 6 bits/char
random.seed(42)
simple_message = ''.join([chr(random.randint(0, 64)) for _ in range(5000)])
h = H_bits(simple_message)
print(f'实测熵 H = {h:.4f} bits/char（理论上限 log2(65) ≈ {math.log(65, 2):.4f}）')
checks.assert_true('2.E6 H 接近 log2(N)', abs(h - math.log(65, 2)) < 0.3)

---

## §3 图模型 (Graphical Models)

**警告**：本节是「先打照面」级别——花书 Ch 16 才完整讲图模型。现在的目标只是知道**贝叶斯网 / 马尔可夫网长什么样、为什么需要它们**，以及**因子分解**这个核心思想（直接通向 Day 4 nanoGPT 的自回归概率分解）。


#### 📖 花书原文 — §3 图模型引入

机器学习算法会涉及到非常多的随机变量上的概率分布。利用分解可以减少表示联合分布的成本，于是用图来表示概率分布的分解，这称为**结构化概率模型 (Structured Probabilistic Model)** 或者**图模型 (Graphical Model)**。

### 为什么需要图模型？

假设你有 30 个二值随机变量。要写出完整的联合分布 $P(x_1, \ldots, x_{30})$，需要 $2^{30} \approx 10^9$ 个参数。**完全无法处理**。

但**很多变量之间没有直接依赖关系**。利用条件独立性，可以把联合分布**分解**成局部因子的乘积，参数量从指数级降到线性级。**图就是这种条件独立结构的画法**。


#### 📖 花书原文 — §3.1 有向图模型 (Directed Model) — 贝叶斯网

有向图模型的概率可以因子分解 $P(x) = P(x_1, \ldots, x_i, \ldots) = \prod_i P(x_i \mid \mathrm{PA}(x_i))$，其中 $\mathrm{PA}(x_i)$ 是 $x_i$ 的父节点，单个因子 $P(x_i \mid \mathrm{PA}(x_i))$ 称为**条件概率分布 (CPD)**。示例如下图所示，有：

$$P(a, b, c, d, e) = P(a) P(b \mid a) P(c \mid a, b) P(d \mid b) P(e \mid c)$$

（图 1：贝叶斯网示例，节点 a→b, a→c, b→c, b→d, c→e）

**有向图的代表是贝叶斯网。**

贝叶斯网与朴素贝叶斯模型建立在相同的直观假设上：**通过利用分布的条件独立性来获得紧凑而自然的表示。**贝叶斯网核心是一个**有向无环图 (DAG)**，其节点为论域中的随机变量，节点间的有向箭头表示这两个节点的依赖关系。

贝叶斯网可以看作是**各特征节点间的依赖关系图**（有向无环图表示）和**各特征节点相对其依赖节点的条件概率表**。

### 📍 Workshop 钩子（重要！）

**链式贝叶斯网 ⟺ 自回归语言模型**。Day 4 你写 nanoGPT 时，把一句话的概率分解成：

$$P(w_1, w_2, \ldots, w_n) = P(w_1) \cdot P(w_2 \mid w_1) \cdot P(w_3 \mid w_1, w_2) \cdots P(w_n \mid w_1, \ldots, w_{n-1})$$

这就是一个**线性链贝叶斯网**——每个 token 节点只指向后面所有 token。
「causal mask（因果掩码）」在 self-attention 里阻止 token 看到未来——本质就是强制这种 DAG 结构。


### §3.1.1 贝叶斯网的独立性

**局部独立性**：给定父节点条件下，每个节点都独立于它的非后代节点。例如给定父节点 $c$ 时，$e$ 与网中其他节点条件独立（$e \perp a, b, d \mid c$）。

**全局独立性 (d-分离)**：d-分离是用来判断变量是否条件独立的图形化方法。常见于三种条件独立的情况：

**1. tail-to-tail（共同原因）** `a ← c → b`
- 不观察 $c$：$P(a, b) = \sum_c P(a \mid c) P(b \mid c) P(c) \neq P(a) P(b)$ → 不独立
- **观察 $c$**：$P(a, b \mid c) = P(a \mid c) P(b \mid c)$ → 条件独立 ✓

**2. head-to-tail（链）** `a → c → b`
- 不观察 $c$：不独立
- **观察 $c$**：条件独立 ✓

**3. head-to-head（V 型 / 碰撞节点）** `a → c ← b`
- **不观察 $c$**：$P(a, b) = P(a) P(b)$ → 独立 ✓（关键！「碰撞节点不观察就独立」）
- 观察 $c$：$P(a, b \mid c) \neq P(a \mid c) P(b \mid c)$ → 反而**不独立**（「explain away 现象」）

**通用 d-分离判别**（对集合 $A, B, C$ 是否条件独立）：
考虑图中所有 $A$ 和 $B$ 之间的路径。如果路径中存在 $X$：
1. $X$ 是 head-to-tail 或 tail-to-tail，且 $X \in C$ → 该路径阻塞
2. $X$ 是 head-to-head，且 $X$ 或 $X$ 的儿子**不在** $C$ 中 → 该路径阻塞

如果 $A, B$ 间所有路径都阻塞 → $A, B$ 关于 $C$ 条件独立。


### 朱明超原文：用 pgmpy 建一个 5 节点贝叶斯网

**注意 API 变化**：朱明超原文是 2020 年代码，用 `pgmpy.models.BayesianModel`。pgmpy 1.0+ 改名为 `DiscreteBayesianNetwork`（功能一样）——我们用新名字。


In [ ]:
import networkx as nx
from pgmpy.models import DiscreteBayesianNetwork   # 朱原文是 BayesianModel
from pgmpy.factors.discrete import TabularCPD

# 建立一个简单贝叶斯模型框架：a→b, a→c, b→c, b→d, c→e
model = DiscreteBayesianNetwork([('a', 'b'), ('a', 'c'), ('b', 'c'), ('b', 'd'), ('c', 'e')])

# 最顶层 a 的先验
cpd_a = TabularCPD(variable='a', variable_card=2, values=[[0.6], [0.4]])   # a: (0,1)
# b 的条件概率：给定 a 的取值（行 = b 值，列 = a 值）
cpd_b = TabularCPD(variable='b', variable_card=2,
                   values=[[0.75, 0.1],
                           [0.25, 0.9]],
                   evidence=['a'], evidence_card=[2])
# c 的条件概率：给定 (a, b) 联合
cpd_c = TabularCPD(variable='c', variable_card=3,
                   values=[[0.3, 0.05, 0.9,  0.5],
                           [0.4, 0.25, 0.08, 0.3],
                           [0.3, 0.7,  0.02, 0.2]],
                   evidence=['a', 'b'], evidence_card=[2, 2])
cpd_d = TabularCPD(variable='d', variable_card=2,
                   values=[[0.95, 0.2],
                           [0.05, 0.8]],
                   evidence=['b'], evidence_card=[2])
cpd_e = TabularCPD(variable='e', variable_card=2,
                   values=[[0.1, 0.4, 0.99],
                           [0.9, 0.6, 0.01]],
                   evidence=['c'], evidence_card=[3])

model.add_cpds(cpd_a, cpd_b, cpd_c, cpd_d, cpd_e)
print('模型一致性检验:', model.check_model())

# 画图
plt.figure(figsize=(7, 5))
nx.draw(model, with_labels=True, node_size=1500, node_color='lightyellow',
        font_weight='bold', font_size=14, edge_color='gray',
        pos={'a': (2, 5), 'b': (5, 5), 'c': (3, 3), 'd': (7, 3), 'e': (3, 1)})
plt.title('贝叶斯网示例（朱明超原文图 1）')
plt.show()

### ✏️ 例题 3.E1：手算因子分解 P(a, b, c)

用上面建好的贝叶斯网（无 $d, e$），手算 $P(a=0, b=0, c=0)$（按链式分解）。

**公式**：$P(a=0, b=0, c=0) = P(a=0) \cdot P(b=0 \mid a=0) \cdot P(c=0 \mid a=0, b=0)$


In [ ]:
# 从 CPT 表里读出对应的条件概率值
P_a0 = 0.6                          # cpd_a values[[0.6], [0.4]]，a=0 的概率
P_b0_given_a0 = 0.75                # cpd_b values[0][0]，给定 a=0 时 b=0
P_c0_given_a0b0 = 0.3               # cpd_c values[0][0]，给定 (a=0,b=0) 时 c=0

P_abc = P_a0 * P_b0_given_a0 * P_c0_given_a0b0
print(f'P(a=0, b=0, c=0) = {P_abc:.4f}')
# 0.6 * 0.75 * 0.3 = 0.135
checks.assert_close('3.E1 P(a=0,b=0,c=0)', P_abc, 0.135)

### ✏️ 例题 3.E2：用 pgmpy 做条件独立查询

**任务**：用 pgmpy 检查 「$a$ 和 $d$ 是否在给定 $e$ 时条件独立」（朱明超原文的例子）。

**朱原文的人脑分析**：从 a 到 d 有两条路径——
- `a→b→d`：$b$ 是 head-to-tail，**不在** $e$ 的集合中 → 不阻塞
- `a→c→b→d`：$c$ 是 head-to-tail 不在集合中，$b$ 也不在集合中 → 不阻塞

所以 a 和 d **不是**关于 e 条件独立的。


In [ ]:
# pgmpy 的 d-separation API
print('a ⊥ d | {} ?:', model.is_dconnected('a', 'd', observed=[]))   # True = 不独立
print('a ⊥ d | {b} ?:', not model.is_dconnected('a', 'd', observed=['b']))
print('a ⊥ d | {e} ?:', not model.is_dconnected('a', 'd', observed=['e']))

# 给定 b 阻塞了 a-b-d 路径，且 a-c-b-d 路径里 b 也阻塞 → 条件独立
indep_given_b = not model.is_dconnected('a', 'd', observed=['b'])
checks.assert_true('3.E2 a ⊥ d | b', indep_given_b)
# 给定 e 不阻塞（朱明超原文结论）→ 不独立
indep_given_e = not model.is_dconnected('a', 'd', observed=['e'])
checks.assert_true('3.E2 a 和 d 给定 e 不独立', not indep_given_e)

#### 📖 花书原文 — §3.2 无向图模型 (Undirected Model) — 马尔可夫网

无向图模型的概率可以记作 $P(\boldsymbol{x}) = \frac{1}{Z} \prod_{C \in \mathbf{Q}} \Phi_C(\boldsymbol{x}_C)$。其中，我们将所有节点都彼此联通的集合称作**团 (Clique, C)**，$\Phi$ 称作**因子 (factor)**，每个因子和一个团 C 相对应，Z 是归一化常数。示例：$P(a, b, c, d, e) = \frac{1}{Z} \Phi^{(1)}(a, b, c) \Phi^{(2)}(b, d) \Phi^{(3)}(c, e)$。

**有向图的代表是马尔可夫网**（错——这是朱原文笔误，应该是「无向图的代表」）。

贝叶斯网是根据节点依赖关系构成有向无环图，进而引申出每个节点的条件概率分布来表征其对父节点的依赖。但马尔可夫网节点间的**依赖关系是无向的**（相互平等的关系），无法用条件概率分布来表示，为此引入**极大团**概念，进而为每个极大团引入一个**势函数 (Potential Function)** 作为因子，然后将联合概率分布表示成这些因子的乘积再归一化，归一化常数被称作**配分函数 (Partition Function)**。

**团**：假设一个特征集的任何两个特征都互相关联，那么这个特征集的联合概率分布是无法简化的，我们称这样的特征集为团。

**极大团**：如果一个团不能被其他团包含，那么我们称这个团为极大团。

对于具有 $n$ 个特征变量 $\boldsymbol{x} = (x_1, \ldots, x_n)$ 的马尔可夫网的所有极大团构成的集合 $\mathbf{Q}$，与极大团 $C \in \mathbf{Q}$ 对应的属性变量集合记作 $\boldsymbol{x}_C$：

$$P(\boldsymbol{x}) = \frac{1}{Z} \prod_{C \in \mathbf{Q}} \Phi_C(\boldsymbol{x}_C), \quad Z = \sum_{\boldsymbol{x}} \prod_{C \in \mathbf{Q}} \Phi_C(\boldsymbol{x}_C)$$

势函数可以写作 $\Phi(\boldsymbol{x}_C) = \exp(-E(\boldsymbol{x}_C))$，其中 $E$ 为**能量函数**，我们也称 $P(\boldsymbol{x})$ 是由因子集 $\{\Phi_C \mid C \in \mathbf{Q}\}$ 参数化的**吉布斯分布 (Gibbs Distribution)** 或**玻尔兹曼分布 (Boltzmann Distribution)**。

**马尔可夫网的条件独立性**：
- **局部马尔可夫性**：将节点 $v$ 的所有邻接节点集作为分离集 $N(v)$，则节点 $v$ 与被邻接变量集分离的剩余变量集是条件独立的：$x_v \perp \boldsymbol{x}_{V \setminus N^*(v)} \mid \boldsymbol{x}_{N(v)}$
- **成对马尔可夫性**：两个非邻接节点 $u, v$，必然可以被其他所有节点构成的集 $\boldsymbol{x}_{V \setminus \{u, v\}}$ 分离，进而 $u, v$ 也具有条件独立性：$x_u \perp x_v \mid \boldsymbol{x}_{V \setminus \{u, v\}}$

In [ ]:
# 朱明超原文马尔可夫网示例：a-b, a-c, b-c, b-d, c-e（无向边）
from pgmpy.models import DiscreteMarkovNetwork   # 朱原文是 MarkovModel；pgmpy 1.1+ 用 Discrete 前缀
from pgmpy.factors.discrete import DiscreteFactor

model_mn = DiscreteMarkovNetwork([('a', 'b'), ('a', 'c'), ('b', 'c'), ('b', 'd'), ('c', 'e')])

# 各团因子（势函数）：随机参数
np.random.seed(0)
factor_abc = DiscreteFactor(['a', 'b', 'c'], cardinality=[2, 2, 2], values=np.random.rand(8))
factor_bd  = DiscreteFactor(['b', 'd'],      cardinality=[2, 2],    values=np.random.rand(4))
factor_ce  = DiscreteFactor(['c', 'e'],      cardinality=[2, 2],    values=np.random.rand(4))
model_mn.add_factors(factor_abc, factor_bd, factor_ce)
print('马尔可夫网一致性:', model_mn.check_model())

plt.figure(figsize=(7, 5))
nx.draw(model_mn, with_labels=True, node_size=1500, node_color='lightyellow',
        font_weight='bold', font_size=14, edge_color='gray',
        pos={'a': (2, 5), 'b': (5, 5), 'c': (3, 3), 'd': (7, 3), 'e': (3, 1)})
plt.title('马尔可夫网示例（朱明超原文图 3）')
plt.show()

### ✏️ 例题 3.E3：识别极大团

上面这个无向图（边集：a-b, a-c, b-c, b-d, c-e），有几个极大团？分别是哪些？

**先在纸上画一下**：哪些节点彼此完全连通？


In [ ]:
# 让 networkx 直接找极大团
G = nx.Graph([('a', 'b'), ('a', 'c'), ('b', 'c'), ('b', 'd'), ('c', 'e')])
max_cliques = list(nx.find_cliques(G))
print('所有极大团:')
for c in max_cliques:
    print(f'  {sorted(c)}')
# 应得: [a, b, c]（三角形）, [b, d]（边）, [c, e]（边）
checks.assert_true('3.E3 共 3 个极大团', len(max_cliques) == 3)
checks.assert_true('3.E3 三角形 {a,b,c} 在内',
                   any(set(c) == {'a', 'b', 'c'} for c in max_cliques))

### 收尾：图模型在花书后续的角色

本节只是**点了个名**。完整的图模型主题在花书 Ch 16 才展开。现在你应该带走的核心 takeaways：

1. **联合分布因子分解**是图模型的灵魂 —— 把指数级参数降到线性
2. **DAG → 贝叶斯网**：箭头表示因果/依赖；自回归语言模型本质就是链式贝叶斯网
3. **无向图 → 马尔可夫网**：用势函数 + 配分函数；玻尔兹曼机 / 能量模型属于这类
4. **d-分离的三种结构**：tail-to-tail、head-to-tail、head-to-head — 记住「碰撞节点不观察才独立」最反直觉

📍 **Workshop 之后会再见**：扩散模型（forward = 链式马尔可夫，每步加 Gaussian 噪声）、VAE 的 latent 变量图模型。


---

## CHECKPOINT — 本章自检

在下面打勾（双击 cell 进编辑模式，把 `[ ]` 改成 `[x]`）：

**§1 概率**

- [ ] 能用一句话说清频率派 vs 贝叶斯派的差别
- [ ] 知道 PMF 输出是概率，PDF 输出是密度（可 > 1），区间概率要积分
- [ ] 给一张联合分布表，能秒算边缘 + 条件 + 判断独立性
- [ ] 能默写期望 / 方差 / 协方差定义；知道 `np.var(ddof=0/1)` 和 `np.cov` 默认 ddof 的区别
- [ ] 知道协方差 = 0 不蕴含独立（构造反例 $y = x^2$）

**七大分布（§1.5）**

- [ ] 看到分布名能回忆形状 + 用途：Bernoulli / Multinoulli / Gaussian / 多元 Gaussian / Exponential / Laplace / Dirac
- [ ] 能手写**数值稳定的 softmax**（先减 max 再 exp）
- [ ] 知道经验分布 $\hat{p}_{\text{data}}$ = 一堆 Dirac 函数的等权和

**§1.6 函数**

- [ ] 能默写 $\sigma(x) = 1/(1+e^{-x})$ 和 $\zeta(x) = \log(1+e^x)$
- [ ] 知道关键性质 $\sigma(-x) = 1 - \sigma(x)$ 和 $\zeta'(x) = \sigma(x)$

**§2 信息论（最重要）**

- [ ] **能默写自信息 $I(x) = -\log P(x)$ + 香农熵 $H(P) = -\mathbb{E}_P[\log P]$**
- [ ] **能默写 $D_{\mathrm{KL}}(P\|Q) = \mathbb{E}_P[\log P - \log Q]$ + 交叉熵 $H(P,Q) = H(P) + D_{\mathrm{KL}}(P\|Q)$**
- [ ] 知道 KL 不对称：正向 KL（覆盖型）vs 反向 KL（挑峰型）
- [ ] 知道 RLHF 的 β·KL penalty 角色 + 「MLE = 最小化交叉熵」连接（**Workshop Day 1-2 立刻用到**）

**§3 图模型**

- [ ] 能默写贝叶斯网因子分解公式 $P(x) = \prod_i P(x_i \mid \mathrm{PA}(x_i))$
- [ ] 知道 d-分离的三种结构 + 「碰撞节点不观察才独立」反直觉点
- [ ] 能把自回归 LM 看作链式贝叶斯网


In [ ]:
checks.report()